# 🔍 RAG + Query Rewriting — LangChain + Ollama + ChromaDB

---

## 🧠 What's New Here? — Query Rewriting

This notebook builds on **Basic RAG** and adds one powerful upgrade: **Query Rewriting**.

**The Problem with Raw User Questions:**  
Users don't always phrase questions the way your documents are written.

| User asks... | Document says... | Match? |
|---|---|---|
| *"Can I take days off?"* | *"Annual leave entitlement policy"* | ❌ Poor |
| *"annual leave entitlement"* | *"Annual leave entitlement policy"* | ✅ Good |

**The Fix — Query Rewriting:**  
Before searching the vector store, use the LLM itself to rephrase the question into an optimal *search query*.

**Analogy:**  
Think of a librarian. You walk in and say *"Do you have anything about people who sail around the world?"*  
A good librarian translates that into *"circumnavigation + maritime exploration"* before searching the catalog.  
Query Rewriting makes your RAG system that smart librarian.

---

## 🗺️ Full Pipeline Overview

```
┌─────────────────────────────────────────────────────────────┐
│                   INDEXING PHASE (run once)                 │
│                                                             │
│  Raw Text  →  TextLoader  →  Chunks  →  Embeddings  →  ChromaDB │
└─────────────────────────────────────────────────────────────┘
              ↓  (query time — the NEW flow)
┌─────────────────────────────────────────────────────────────┐
│                   QUERYING PHASE (per question)             │
│                                                             │
│  User Question (raw, informal)                              │
│         │                                                   │
│  [Step 7] Query Rewriter (LLM)                              │
│         │  → Rephrases into an optimized search query       │
│         │                                                   │
│  [Step 8a] Retrieve using ORIGINAL question  (for compare) │
│  [Step 8b] Retrieve using REWRITTEN query    (for answer)  │
│         │                                                   │
│  [Step 8c] Build prompt with rewritten context             │
│         │                                                   │
│  [Step 8d] LLM generates grounded answer                   │
└─────────────────────────────────────────────────────────────┘
```

> 💡 **Key Insight:** The LLM is used *twice* — once to rewrite the query, once to generate the answer.  
> The original question is still shown to the user in the final answer for transparency.

## 📦 Prerequisites & Installation

Same as Basic RAG. Ensure the following are ready:

1. **Ollama running** → https://ollama.com
2. **Models pulled:**
   ```bash
   ollama pull nomic-embed-text
   ollama pull gpt-oss:120b-cloud
   ```
3. **Packages installed:**
   ```bash
   pip install langchain langchain-community langchain-ollama langchain-chroma langchain-core chromadb
   ```
4. **Sample file at** `docs/company_policy.txt`

---
## 📚 Step 0 — Imports

One new import compared to Basic RAG:

| Import | Role |
|---|---|
| `TextLoader` | Reads `.txt` file from disk |
| `RecursiveCharacterTextSplitter` | Splits text into overlapping chunks |
| `OllamaEmbeddings` | Converts text → dense vectors |
| `Chroma` | Local vector database |
| `ChatOllama` | Local LLM for generation |
| `PromptTemplate` ⭐ NEW | Reusable, parameterised prompt builder for the rewriter |

In [1]:
# ── Document Ingestion ──────────────────────────────────────────────────────
from langchain_community.document_loaders.text import TextLoader

# ── Chunking ────────────────────────────────────────────────────────────────
from langchain_text_splitters import RecursiveCharacterTextSplitter

# ── Embeddings ──────────────────────────────────────────────────────────────
from langchain_ollama import OllamaEmbeddings

# ── Vector Store ────────────────────────────────────────────────────────────
from langchain_chroma import Chroma

# ── LLM ─────────────────────────────────────────────────────────────────────
from langchain_ollama import ChatOllama

# ── ⭐ NEW: PromptTemplate — structures the rewrite instruction for the LLM ──
from langchain_core.prompts import PromptTemplate

print("✅ All imports successful")

C:\Users\sr43993\AppData\Local\Temp\ipykernel_22556\2302372665.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders.text import TextLoader


✅ All imports successful


---
## 📄 Steps 1–6 — Indexing Pipeline

These steps are identical to Basic RAG:
1. Load document
2. Split into chunks
3. Create embedding model
4. Build & persist ChromaDB vector store
5. Create retriever
6. Load LLM

> 📖 Refer to the **Basic RAG notebook** for deep-dive explanations of each step below.

In [2]:
# ── Step 1: Load Document ────────────────────────────────────────────────────
print("Loading document...\n")

loader = TextLoader(
    "docs/company_policy.txt",
    encoding="utf-8"
)
documents = loader.load()

print(f"✅ Loaded {len(documents)} document(s)")
print(f"\n📄 Preview: {documents[0].page_content[:200]}")

Loading document...

✅ Loaded 1 document(s)

📄 Preview: Employees are eligible for 6 months maternity leave.

Employees must serve 60 days notice period before resignation.

Annual leave balance is 24 days.

Work from home is allowed twice a week.


In [3]:
# ── Step 2: Split into Chunks ────────────────────────────────────────────────
print("Splitting document into chunks...\n")

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=200,
    chunk_overlap=50
)
chunks = text_splitter.split_documents(documents)

print(f"✅ Total Chunks Created: {len(chunks)}")
print(f"\n--- First 3 chunks ---")
for i, chunk in enumerate(chunks[:3]):
    print(f"\n🔹 Chunk {i+1} ({len(chunk.page_content)} chars): {chunk.page_content}")
    print("-" * 50)

Splitting document into chunks...

✅ Total Chunks Created: 1

--- First 3 chunks ---

🔹 Chunk 1 (191 chars): Employees are eligible for 6 months maternity leave.

Employees must serve 60 days notice period before resignation.

Annual leave balance is 24 days.

Work from home is allowed twice a week.
--------------------------------------------------


In [4]:
# ── Step 3: Embedding Model ──────────────────────────────────────────────────
print("Loading Embedding Model...\n")

embeddings = OllamaEmbeddings(model="nomic-embed-text")

print("✅ Embedding Model Ready")

# Sanity check
test_vec = embeddings.embed_query("test")
print(f"🔢 Vector dimension: {len(test_vec)}")

Loading Embedding Model...

✅ Embedding Model Ready
🔢 Vector dimension: 768


In [5]:
# ── Step 4: Create & Persist Vector Store ────────────────────────────────────
print("Creating Chroma Vector Store...\n")

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory="./chroma_db"  # Persists to disk — reuse without re-indexing
)

print(f"✅ Vector Store Ready — {vectorstore._collection.count()} vectors stored")

Creating Chroma Vector Store...

✅ Vector Store Ready — 4 vectors stored


In [6]:
# ── Step 5: Create Retriever ─────────────────────────────────────────────────
# k=2 → return the 2 most semantically similar chunks per query

retriever = vectorstore.as_retriever(
    search_kwargs={"k": 2}
)

print("✅ Retriever Ready")

✅ Retriever Ready


In [7]:
# ── Step 6: Load LLM ─────────────────────────────────────────────────────────
print("Loading LLM...\n")

llm = ChatOllama(
    model="gpt-oss:120b-cloud",
    temperature=0   # Must be 0 for RAG — deterministic, grounded answers only
)

print("✅ LLM Ready")

Loading LLM...

✅ LLM Ready


---
## ✏️ Step 7 — Build the Query Rewriter

This is the **core new concept** in this notebook.

### What is `PromptTemplate`?

`PromptTemplate` is LangChain's way of creating **reusable, parameterized prompts** — like a function where `{question}` is a variable that gets filled in at runtime.

```python
# Think of it like an f-string, but with superpowers:
# — It validates that all required variables are provided
# — It integrates cleanly into LangChain chains
# — It's easy to version and swap in production
```

### What is `rewrite_chain = rewrite_prompt | llm`?

This is the **LangChain Expression Language (LCEL)** pipe operator `|` — it chains two components so data flows left to right:

```
rewrite_prompt   |   llm
     ↓                ↓
Fill {question}  →  Send to LLM  →  Get rewritten query back
```

**Analogy:**  
It's exactly like Unix pipes: `cat file.txt | grep "error"` — output of one step becomes input of the next.

> ⚠️ **Common Mistake:** Using a creative temperature for the rewriter.  
> The LLM is already loaded with `temperature=0`, so the rewritten query is consistent and predictable — exactly what you want for a search query.

> 💡 **Production Note:** You could use a *smaller, faster* model (e.g. `llama3.2:3b`) for the rewriter and reserve the large model only for the final answer generation. This cuts latency significantly.

In [8]:
# ── Query Rewrite Prompt ──────────────────────────────────────────────────────
# {question} is a placeholder — filled at runtime via .invoke({"question": ...})

rewrite_prompt = PromptTemplate.from_template(
    """
You are a search expert.

Rewrite the user's question into a concise search query
that would help retrieve the most relevant documents.

Return ONLY the rewritten search query — no explanation, no preamble.

Question:
{question}

Search Query:
"""
)

# ── Build the Rewrite Chain using LCEL pipe operator ─────────────────────────
# Flow: fill prompt template → send to LLM → return rewritten query string
rewrite_chain = rewrite_prompt | llm

print("✅ Query Rewriter Ready")

# ── Quick test — see what the rewriter does to a sample question ──────────────
sample_question = "Can I take days off when I'm sick?"
sample_rewrite = rewrite_chain.invoke({"question": sample_question})

print(f"\n🧪 Sample Rewrite Test:")
print(f"  Original  : {sample_question}")
print(f"  Rewritten : {sample_rewrite.content.strip()}")

✅ Query Rewriter Ready

🧪 Sample Rewrite Test:
  Original  : Can I take days off when I'm sick?
  Rewritten : sick leave policy taking days off when ill


---
## 💬 Step 8 — Interactive Q&A Loop with Query Rewriting

### The Full Flow for Every Question:

```
User's raw question
        │
        ├──────────────────────────────────────────────────┐
        │                                                  │
   [Rewrite Chain]                                  [kept for display]
   LLM rewrites to                                  shown in final
   optimized search query                           answer to user
        │
        ├── [Retrieval A] Original question → ChromaDB  (comparison)
        ├── [Retrieval B] Rewritten query  → ChromaDB  (used for answer)
        │
   Build final prompt:
   context from Retrieval B + original question
        │
   LLM generates grounded answer
```

### Why retrieve with BOTH original AND rewritten?

This notebook does **side-by-side comparison** so you can *see the difference* in retrieved chunks — ideal for learning and debugging. In production, you'd typically only use the rewritten query retrieval.

In [9]:
print("RAG + Query Rewriting System Ready. Type 'exit' to stop.\n")
print("=" * 60)

while True:

    question = input("\n❓ Ask a question (or type 'exit'): ")

    if question.lower() == "exit":
        print("\n👋 Goodbye!")
        break

    # ── STEP 8a: REWRITE THE QUERY ────────────────────────────────────────────
    # The LLM rephrases the user's informal question into a precise search query
    rewritten_response = rewrite_chain.invoke({"question": question})
    rewritten_query_text = rewritten_response.content.strip()

    print("\n===== ✍️  ORIGINAL QUESTION =====")
    print(question)

    print("\n===== 🔄 REWRITTEN QUERY =====")
    print(rewritten_query_text)

    # ── STEP 8b: RETRIEVE USING ORIGINAL QUESTION ────────────────────────────
    # For comparison only — lets you see what raw retrieval would have returned
    original_docs = retriever.invoke(question)

    print("\n===== 📄 ORIGINAL RETRIEVAL (for comparison) =====")
    for index, doc in enumerate(original_docs):
        print(f"\n  Chunk {index + 1}: {doc.page_content}")

    # ── STEP 8c: RETRIEVE USING REWRITTEN QUERY ──────────────────────────────
    # This is what actually feeds into the final answer
    retrieved_docs = retriever.invoke(rewritten_query_text)

    print("\n===== 📚 REWRITTEN RETRIEVAL (used for answer) =====")
    for index, doc in enumerate(retrieved_docs):
        print(f"\n  Chunk {index + 1}: {doc.page_content}")

    # ── STEP 8d: BUILD CONTEXT FROM REWRITTEN RETRIEVAL ──────────────────────
    # Join all retrieved chunks; these become the LLM's knowledge source
    context = "\n\n".join(
        doc.page_content for doc in retrieved_docs
    )

    # ── STEP 8e: CRAFT THE FINAL ANSWER PROMPT ───────────────────────────────
    # Note: we pass the ORIGINAL question here (not the rewritten one)
    # The rewriting was purely for better retrieval — the user's intent stays intact
    prompt = f"""You are a helpful assistant.

Answer ONLY using the provided context.
If the answer is not present in the context, reply exactly with:
"I could not find that information in the document."

Context:
{context}

Question:
{question}
"""

    # ── STEP 8f: GENERATE ANSWER ──────────────────────────────────────────────
    response = llm.invoke(prompt)

    print("\n===== 🤖 ANSWER =====")
    print(response.content)

RAG + Query Rewriting System Ready. Type 'exit' to stop.


===== ✍️  ORIGINAL QUESTION =====
maternity leave

===== 🔄 REWRITTEN QUERY =====
maternity leave policies

===== 📄 ORIGINAL RETRIEVAL (for comparison) =====

  Chunk 1: Employees are eligible for 6 months maternity leave.

Employees must serve 60 days notice period before resignation.

Annual leave balance is 24 days.

Work from home is allowed twice a week.

  Chunk 2: Employees are eligible for 6 months maternity leave.

Employees must serve 60 days notice period before resignation.

Annual leave balance is 24 days.

Work from home is allowed twice a week.

===== 📚 REWRITTEN RETRIEVAL (used for answer) =====

  Chunk 1: Employees are eligible for 6 months maternity leave.

Employees must serve 60 days notice period before resignation.

Annual leave balance is 24 days.

Work from home is allowed twice a week.

  Chunk 2: Employees are eligible for 6 months maternity leave.

Employees must serve 60 days notice period before res

---
## 🧪 Bonus — Single-Shot Query (No Loop)

Test any question without entering the interactive loop. Great for notebook demos.

In [10]:
# ── Change this question and re-run to test ───────────────────────────────────
question = "What happens if I'm late to work repeatedly?"

# Rewrite
rewritten = rewrite_chain.invoke({"question": question}).content.strip()

# Retrieve using rewritten query
docs = retriever.invoke(rewritten)
context = "\n\n".join(doc.page_content for doc in docs)

# Generate answer
prompt = f"""Answer ONLY using the context below.
If not found, say: "I could not find that information in the document."

Context:
{context}

Question: {question}
"""
answer = llm.invoke(prompt).content

print(f"❓ Original Question : {question}")
print(f"🔄 Rewritten Query   : {rewritten}")
print(f"\n🤖 Answer: {answer}")

❓ Original Question : What happens if I'm late to work repeatedly?
🔄 Rewritten Query   : consequences of repeated tardiness at work disciplinary policies

🤖 Answer: I could not find that information in the document.


---
## 🔬 Bonus — Side-by-Side Rewrite Comparison

Run multiple questions at once to see how the rewriter transforms them.  
This cell is **purely diagnostic** — no answer generation.

In [11]:
test_questions = [
    "Can I take days off?",
    "What if I need to leave early?",
    "Is there a dress code?",
    "Who do I tell if I'm sick?",
]

print(f"{'Original Question':<45} {'Rewritten Query'}")
print("-" * 90)

for q in test_questions:
    rewritten = rewrite_chain.invoke({"question": q}).content.strip()
    print(f"{q:<45} {rewritten}")

Original Question                             Rewritten Query
------------------------------------------------------------------------------------------
Can I take days off?                          employee days off policy
What if I need to leave early?                policy for leaving early requirements
Is there a dress code?                        dress code policy
Who do I tell if I'm sick?                    who to notify when sick


---
## 🎯 Key Takeaways

| Concept | What to Remember |
|---|---|
| **Query Rewriting** | Bridges the gap between how users *ask* and how documents *describe* |
| **`PromptTemplate`** | Reusable, parameterized prompt — like a function for LLM instructions |
| **LCEL `\|` operator** | Chains LangChain components left-to-right — clean, composable pipelines |
| **Original question preserved** | Rewriting is only for retrieval — the answer always addresses the original intent |
| **Dual retrieval (this notebook)** | Side-by-side comparison is for *learning* — in production, use rewritten only |
| **Smaller model for rewriting** | In production, use a fast 3B model to rewrite and a large model to answer |

---

## 🔄 Basic RAG vs RAG + Query Rewriting

```
Basic RAG:
  User question ──────────────────→ Retriever → LLM → Answer

RAG + Query Rewriting:
  User question → LLM (rewrite) → Retriever → LLM → Answer
                  ↑
            Extra LLM call — small cost, big retrieval improvement
```

---

## 🚀 Next Steps to Explore

1. **Multi-query retrieval** — Generate 3–5 rewritten variants of the same question and merge results for broader coverage
2. **HyDE (Hypothetical Document Embeddings)** — Ask the LLM to generate a *hypothetical answer*, embed that, and use it for retrieval
3. **Step-back prompting** — Rewrite the question into a more general/abstract form to retrieve broader context first
4. **Reranking** — After retrieval, use a cross-encoder to reorder chunks by actual relevance
5. **RAGAS evaluation** — Measure whether rewriting actually improves faithfulness and context recall scores